**BBM473 Database Laboratory (Spring 2025)**


Exercise 6: Indexes
===========

Let's play with the [consumer complaint database](https://catalog.data.gov/dataset/consumer-complaint-database) from data.gov

In [35]:
%load_ext sql
%config SqlMagic.style = '_DEPRECATED_DEFAULT'

%sql sqlite:///complaint.db
import time
%sql drop index if exists state_product_index;
%sql analyze

The sql extension is already loaded. To reload it, use:
  %reload_ext sql
 * sqlite:///complaint.db
Done.
 * sqlite:///complaint.db
Done.


[]

In [36]:
%sql select count(*) from complaints;

 * sqlite:///complaint.db
Done.


count(*)
79468


In [37]:
%sql select * from complaints limit 5;

 * sqlite:///complaint.db
Done.


Date_received,Product,Subproduct,Issue,Subissue,Consumer_narrative,Company_public_response,Company,State,ZIP_code,Submitted_via,Date_sent_to_company,Company_response,Timely_response,Consumer_disputed,Complaint_ID
12/30/2015,Mortgage,Other mortgage,"Loan servicing, payments, escrow account",,,Company chooses not to provide a public response,U.S. Bancorp,TN,38138,,N/A,Referral,01/05/2016,Closed with explanation,Yes
12/12/2015,Mortgage,Other mortgage,"Loan modification,collection,foreclosure",,,Company chooses not to provide a public response,Citibank,NY,13021,,N/A,Referral,12/23/2015,Closed with explanation,Yes
12/02/2015,Mortgage,Other mortgage,"Loan modification,collection,foreclosure",,,,Nationstar Mortgage,MI,49102,,N/A,Referral,12/17/2015,Closed with explanation,Yes
12/02/2015,Bank account or service,Other bank product/service,"Account opening, closing, or management",,,Company chooses not to provide a public response,Wells Fargo & Company,,,,N/A,Referral,12/07/2015,Closed with explanation,Yes
12/22/2015,Mortgage,Conventional fixed mortgage,"Loan servicing, payments, escrow account",,,,Nationstar Mortgage,FL,33484,Older American,Consent not provided,Web,01/06/2016,Closed with explanation,Yes


### Task 1: Query without an index (30 pts.)

First, let's start off by writing a query to find the **counts of the top 5 Product, State pairs** in the complaints database (return the product and state as well as the count).  Use the single-line syntax for simple timing so we can see how long the query takes:

In [38]:
%time %sql SELECT Product, State, COUNT(*) AS count  FROM complaints GROUP BY Product, State ORDER BY count DESC LIMIT 5;

 * sqlite:///complaint.db
Done.
CPU times: user 73.8 ms, sys: 27.8 ms, total: 102 ms
Wall time: 102 ms


In [39]:
%sql SELECT Product, State, COUNT(*) AS count  FROM complaints GROUP BY Product, State ORDER BY count DESC LIMIT 5;

 * sqlite:///complaint.db
Done.


Product,State,count
None,None,13451
Mortgage,CA,3891
Mortgage,FL,2343
Debt collection,None,1654
Mortgage,None,1427


### Task 2: Single search key index (30 pts.)

Now create a _single-key_ index such that the above query is faster!  The syntax to create an index in SQL is:
> DROP INDEX IF EXISTS index_name; --- Deletes if an index with the same name exists

> CREATE INDEX index_name ON table(attributes);

In [40]:
%%sql
DROP INDEX IF EXISTS idx_product;
CREATE INDEX idx_product ON complaints(Product);

 * sqlite:///complaint.db
Done.
Done.


[]

In [41]:
%time %sql SELECT Product, State, COUNT(*) AS count FROM complaints GROUP BY Product, State ORDER BY count DESC LIMIT 5;

 * sqlite:///complaint.db
Done.
CPU times: user 73.1 ms, sys: 26.4 ms, total: 99.5 ms
Wall time: 98.7 ms


### Task 3.a: (20 pts.)

Now, create a _covering_ index for the query and then see how long it takes to run!

In [44]:
%%sql 
DROP INDEX IF EXISTS idx_product_state;
CREATE INDEX idx_product_state ON complaints(Product, State);

 * sqlite:///complaint.db
Done.
Done.


[]

In [45]:
%time %sql SELECT Product, State, COUNT(*) AS count FROM complaints GROUP BY Product, State ORDER BY count DESC LIMIT 5;

 * sqlite:///complaint.db
Done.
CPU times: user 9.6 ms, sys: 1.75 ms, total: 11.4 ms
Wall time: 10.7 ms


### Task 3.b: (20 pts.)

Use EXPLAIN to see if sqlite used/recognized your covering index.  EXPLAIN is an operator that tells SQL to explain its query plan... we'll look into this in more depth later.  For now, the syntax is:
> EXPLAIN QUERY PLAN your_query_here;

In [46]:
%%sql
EXPLAIN QUERY PLAN
SELECT Product, State, COUNT(*) AS count
FROM complaints
GROUP BY Product, State
ORDER BY count DESC
LIMIT 5;


 * sqlite:///complaint.db
Done.


id,parent,notused,detail
8,0,0,SCAN complaints USING COVERING INDEX idx_product_state
44,0,0,USE TEMP B-TREE FOR ORDER BY


Save and submit your notebook file (.ipynb) to the form below with your name and student ID. Do not include the database file or any other file. Only one notebook file is sufficient.

https://forms.gle/FSMKdAa5XCqZ2e5W9

**Note:** *You **can** submit your exercise notebook remotely.*